In [1]:
!pip -q install ipywidgets
from google.colab import output; output.enable_custom_widget_manager()
import os, ipywidgets as widgets
from IPython.display import display, clear_output

cid = widgets.Text(description='Client ID:', layout=widgets.Layout(width='70%'))
cs  = widgets.Password(description='Client Secret:', layout=widgets.Layout(width='70%'))
btn = widgets.Button(description='Set secrets', button_style='success')
out = widgets.Output()

def on_click(_):
    os.environ['SLACK_CLIENT_ID'] = cid.value.strip()
    os.environ['SLACK_CLIENT_SECRET'] = cs.value.strip()
    cid.value = ""; cs.value = ""
    with out: clear_output(); print("✅ Secrets set in env (not printed).")

btn.on_click(on_click)
display(widgets.VBox([cid, cs, btn, out]))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 21.2 MB/s eta 0:00:00


In [2]:
!pip -q install slack_sdk requests
import os, urllib.parse, requests
from urllib.parse import urlparse, parse_qs

REDIRECT_URI = "https://oauth.pstmn.io/v1/callback"
USER_SCOPES  = "im:read,im:history,mpim:read,mpim:history,users:read"

authorize_url = "https://slack.com/oauth/v2/authorize?" + urllib.parse.urlencode({
    "client_id": os.environ["SLACK_CLIENT_ID"],
    "user_scope": USER_SCOPES,
    "redirect_uri": REDIRECT_URI
})
print("Open & Allow:\n", authorize_url)

full = input("\nPaste FULL redirect URL: ").strip()
code = parse_qs(urlparse(full).query).get("code", [None])[0]

resp = requests.post("https://slack.com/api/oauth.v2.access", data={
    "client_id": os.environ["SLACK_CLIENT_ID"],
    "client_secret": os.environ["SLACK_CLIENT_SECRET"],
    "code": code,
    "redirect_uri": REDIRECT_URI
}).json()

xoxp = (resp.get("authed_user") or {}).get("access_token") or resp.get("access_token")
os.environ["SLACK_USER_TOKEN"] = xoxp
print("✅ New user token set.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.9/293.9 kB 6.9 MB/s eta 0:00:00
Open & Allow:
 https://slack.com/oauth/v2/authorize?client_id=8048382690262.9605912667138&user_scope=im%3Aread%2Cim%3Ahistory%2Cmpim%3Aread%2Cmpim%3Ahistory%2Cusers%3Aread&redirect_uri=https%3A%2F%2Foauth.pstmn.io%2Fv1%2Fcallback

Paste FULL redirect URL: https://oauth.pstmn.io/v1/callback?code=8048382690262.9607732876674.7da542ba86e46b77fca06514e0448eda979520d90395098dc3f2dcc252e79a43&state=
✅ New user token set.


In [3]:
!pip -q install slack_sdk
from slack_sdk import WebClient
import os
client = WebClient(token=os.environ["SLACK_USER_TOKEN"])
print(client.auth_test())   # expect ok: True, plus team/user ids


{'ok': True, 'url': 'https://abc-s939779.slack.com/', 'team': 'SJSU_MSDA', 'user': 'debika.choudhury27', 'team_id': 'T081EB8LA7Q', 'user_id': 'U0829PC23J4', 'is_enterprise_install': False}


In [4]:
!pip -q install slack_sdk pandas
from slack_sdk import WebClient
import pandas as pd
from datetime import datetime, timezone, timedelta
import os, re

# ---------- CONFIG ----------
TARGET_NAMES = ["ketki", "hiru"]    # case-insensitive
DAYS = 30
OUT_SAFE = "/content/dms_ketki_hirok_mine_final.csv"   # redacted for screenshots
# ----------------------------

# 0) Client (token is never printed)
token = os.getenv("SLACK_USER_TOKEN")
assert token and token.startswith(("xoxp","xoxa")), "Set SLACK_USER_TOKEN first (xoxp- user token)."
client = WebClient(token=token)

# 1) Identity proof (safe to show: no tokens here)
me = client.auth_test()
print("Auth OK:", bool(me.get("ok")), "| Team:", me.get("team"), "| User:", me.get("user"))

# 2) Build user map (id -> display name)
users, cursor = [], None
while True:
    r = client.users_list(limit=200, cursor=cursor)
    users += r["members"]
    cursor = r.get("response_metadata", {}).get("next_cursor")
    if not cursor: break

def disp(u):
    prof = (u.get("profile") or {})
    return prof.get("real_name") or u.get("name") or u["id"]

user_map = {u["id"]: disp(u) for u in users}
my_user_id = me["user_id"]

# 3) List 1:1 DMs and keep only Ketki/Hirok
def list_ims():
    ims, cur = [], None
    while True:
        resp = client.conversations_list(types="im", limit=1000, cursor=cur)
        ims += resp["channels"]
        cur = resp.get("response_metadata", {}).get("next_cursor")
        if not cur: break
    return ims

ims = list_ims()
def match_name(uid):
    name = user_map.get(uid, "").lower()
    return any(t.lower() in name for t in TARGET_NAMES)

target_ims = [im for im in ims if match_name(im.get("user"))]
assert target_ims, "No 1:1 DMs found for: " + ", ".join(TARGET_NAMES)

# 4) Fetch last N days of history for those IMs
def fetch_history(channel_id, days=DAYS):
    rows, cursor = [], None
    oldest = str((datetime.now(tz=timezone.utc) - timedelta(days=days)).timestamp())
    while True:
        resp = client.conversations_history(channel=channel_id, oldest=oldest, limit=200, cursor=cursor)
        for m in resp.get("messages", []):
            if m.get("type") != "message": continue
            ts = float(m["ts"])
            sender_id = m.get("user") or m.get("bot_id")
            rows.append({
                "channel_id": channel_id,
                "ts_iso": datetime.fromtimestamp(ts, tz=timezone.utc).isoformat(),
                "sender_id": sender_id,
                "sender_name": user_map.get(sender_id, "unknown"),
                "text": m.get("text",""),
                "dm_with": user_map.get(next(im.get("user") for im in target_ims if im["id"]==channel_id), "unknown"),
            })
        cursor = resp.get("response_metadata", {}).get("next_cursor")
        if not cursor: break
    return pd.DataFrame(rows)

frames = [fetch_history(im["id"], DAYS) for im in target_ims]
msgs = pd.concat(frames, ignore_index=True)

# 5) Keep only messages YOU sent
mine = msgs[msgs["sender_id"] == my_user_id].copy().sort_values("ts_iso")

# 6) REDACT for display/screenshot: mask IDs + trim text
def mask(s):
    s = str(s)
    return s if len(s) < 8 else f"{s[:4]}…{s[-3:]}"
safe = mine.copy()
safe["channel_id"] = safe["channel_id"].map(mask)
safe["sender_id"]  = safe["sender_id"].map(mask)
safe["text"] = safe["text"].str.replace(r"\s+", " ", regex=True).str.slice(0, 140)  # shorten for slide

print(f"Your messages matched: {len(safe)}")
display(safe[["channel_id","ts_iso","sender_id","sender_name","text","dm_with"]].tail(20))

# 7) Save REDACTED CSV for sharing/screenshots
safe.to_csv(OUT_SAFE, index=False)
print("Saved →", OUT_SAFE)

# (Nothing sensitive was printed. No tokens or client secrets are in the CSV.)


Auth OK: True | Team: SJSU_MSDA | User: debika.choudhury27
Your messages matched: 2


,channel_id,ts_iso,sender_id,sender_name,text,dm_with
1,D09H…T6X,2025-09-30T01:54:16.425019+00:00,U082…3J4,Debika Choudhury,Hi Ketki! Can we meet *tomorrow at 3:00 PM PT*...,ketkimaddiwar94
0,D09H…YRY,2025-09-30T03:53:02.770449+00:00,U082…3J4,Debika Choudhury,"Hi Hirok, Can we have a sync up call tomorrow ...",hiru01


Saved → /content/dms_ketki_hirok_mine_final.csv
